# Index -> Cross-Ticker Signal Assessment

This notebook walks through the complete pipeline for evaluating **pre-aggregated sentiment indices** as predictors of asset price returns across multiple commodities and FX pairs.

## What This Notebook Does

1. **Loads** hourly sentiment index data from zip archives (one per ticker)
2. **Aggregates** hourly indices into daily sentiment per topic, deriving **4 sentiment metrics** from the raw columns
3. **Loads** price data and computes forward returns at multiple horizons (7d, 1m, 3m)
4. **Correlates** all 4 metrics x rolling transforms (z-score, mean) x 3 lookback windows with forward price returns across 3 regimes (ALL, UP market, DOWN market)
5. **Applies** multiple-testing corrections (Benjamini-Hochberg FDR) to identify statistically significant drivers
6. **Validates** signals with a 60/40 walk-forward in-sample / out-of-sample split
7. **Ranks** tickers, sentiment groups, and **metrics** in a cross-asset comparison table

## Sentiment Metrics

The raw hourly data contains 5 columns (`headline_count`, `sentiment_sum`, `sentiment_abs_sum`, `sentiment_std`, `publication_time`). From these we derive **4 daily metrics**, each capturing a different aspect of sentiment:

| Metric | Formula | What it measures |
|---|---|---|
| `sentiment_avg` | `sentiment_sum / headline_count` | Average sentiment direction (bullish/bearish) |
| `sentiment_std_daily` | mean of hourly `sentiment_std` | Intra-day disagreement / dispersion |
| `headline_count` | sum of hourly counts | Attention / news volume |
| `conviction_ratio` | `sentiment_sum / sentiment_abs_sum` | Net directional conviction (-1 to +1) |

**Why these 4?** An initial investigation tested 6 metrics. `sentiment_abs_avg` and `intensity_per_headline` were dropped because they produce identical results to each other (both are `sentiment_abs_sum / headline_count`) and neither outperforms the remaining 4 on any dimension.

**Key finding**: `conviction_ratio` and `sentiment_std_daily` frequently outperform the baseline `sentiment_avg`, producing stronger correlations and often better OOS persistence. `conviction_ratio` is the dominant metric for ~half of tickers, while `sentiment_std_daily` leads for the rest.

## Analysis Parameters

| Parameter | Value | Rationale |
|---|---|---|
| Return horizons | 7d, 1m, 3m | 1d dropped -- 0.9% BH-significance rate (nothing survives correction) |
| Rolling windows | 10, 21, 63 days | w=5 dropped -- lowest significance rate (9.6% vs 12-18%) |
| Transforms | rolling z-score, rolling mean | Z-score is stationary; rolling mean captures trends |
| Regimes | BOTH, UP, DOWN | Based on trailing 63-day return sign |
| Walk-forward | 60% IS / 40% OOS | Single temporal split for persistence validation |
| Significance | BH FDR at q=0.05, per regime | Controls false discovery rate within each regime |

## Input Data

**Sentiment Indices** (`asset_indices/*.zip`): Each zip contains gzipped CSVs partitioned by `index_type` (e.g. COMBINED, EXPLICIT, IMPLICIT) and `topic` (e.g. `Macro & Geopolitics-Central Banking`). Each row is one hourly observation with columns:

| Column | Description |
|---|---|
| `publication_time` | Hourly timestamp |
| `headline_count` | Number of headlines in this hour |
| `sentiment_sum` | Sum of sentiment scores across headlines |
| `sentiment_abs_sum` | Sum of absolute sentiment scores |
| `sentiment_std` | Standard deviation of sentiment scores |

**Price Data** (`prices/prices.csv`): Daily prices with columns `ticker`, `date`, `price`.

## How This Differs from the Headlines Notebook

The **headlines notebook** (`sentiment_headlines_guide.ipynb`) works with **raw headline-level data** and includes a filter-variation step (testing topic confidence, sentiment strength, language filters). This notebook uses **pre-aggregated index data** where filtering has already been applied upstream. As a result:

- No filter-variation step is needed
- The `index_type` partitioning is different (indices use COMBINED/EXPLICIT/IMPLICIT from the index provider, rather than match types derived from headline data)
- `sentiment_std_daily` is computed as the mean of hourly std values (vs computed directly from headline variance in the headlines notebook)
- The correlation analysis, significance testing, walk-forward validation, and cross-ticker ranking steps are identical between the two notebooks

## High-Level Output

- **Overall Ranking**: Each ticker graded A-F based on signal strength, correlation magnitude, and out-of-sample persistence
- **Metric Comparison**: Which of the 4 metrics produces the strongest / most persistent signals per ticker
- **Top Sentiment Drivers Per Ticker**: The single best-scoring sentiment topic for each asset, with the metric that produces it
- **Top Sentiment Group Drivers Per Ticker**: Aggregated by sentiment group (e.g. *Physical Demand*, *Macro & Geopolitics*) showing depth of signal
- **Walk-Forward Validation**: Per-ticker persistence rates, sign consistency, and statistical tests (Fisher-Z, binomial, chi-squared, meta-correlation)
- **Regime Insights**: Which drivers are regime-specific vs regime-universal, and any sign-flipping patterns

## Step 0: Configuration & Imports

Set up paths, tickers to analyse, and the key parameters that control the correlation analysis.

In [ ]:
import warnings, zipfile, gzip, time
from pathlib import Path
from IPython.display import display as ipy_display

import numpy as np
import pandas as pd
from scipy.stats import t as t_dist

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

BASE_DIR = Path(r"g:\My Drive\Code\Asset_Data_Analysis")
ASSET_INDICES_DIR = BASE_DIR / "asset_indices"
PRICES_FILE = BASE_DIR / "prices" / "prices.csv"

TICKERS = ["QO_COM", "QC_COM", "ZS_COM", "AUD_IND", "BZ_COM", "NG_COM", "CL_COM"]

TICKER_LABELS = {
    "QO_COM": "Gold",
    "QC_COM": "Copper",
    "ZS_COM": "Soybeans",
    "AUD_IND": "Australian Dollar",
    "BZ_COM": "Brent Crude",
    "NG_COM": "Natural Gas",
    "CL_COM": "WTI Crude Oil",
}

RETURN_HORIZONS = {"7d": 7, "1m": 21, "3m": 63}
ROLLING_WINDOWS = [10, 21, 63]
SIGNIFICANCE_LEVEL = 0.05
REGIME_LOOKBACK = 63
REGIMES = ["BOTH", "UP", "DOWN"]
SPARSE_THRESHOLD_PCT = 15
WALK_FORWARD_IS_FRACTION = 0.60

SENTIMENT_METRICS = [
    "sentiment_avg",
    "sentiment_std_daily",
    "headline_count",
    "conviction_ratio",
]

print(f"Tickers: {TICKERS}")
print(f"Sentiment metrics: {SENTIMENT_METRICS}")
print(f"Return horizons: {list(RETURN_HORIZONS.keys())}")
print(f"Rolling windows: {ROLLING_WINDOWS}")
print(f"Regimes: {REGIMES}  (lookback={REGIME_LOOKBACK}d)")
print(f"Walk-forward split: {WALK_FORWARD_IS_FRACTION * 100:.0f}% IS / {(1 - WALK_FORWARD_IS_FRACTION) * 100:.0f}% OOS")

## Step 1: Load Sentiment Indices from Zip Files

Each ticker has a zip archive containing gzipped CSV files partitioned by `index_type` and `topic`. Unlike the raw headlines data, these are **pre-aggregated at the hourly level** with headline counts and sentiment sums already computed.

We read all partitions, extract the `index_type` and `topic` from the directory structure, and parse the timestamp. Some tickers may have multiple zip files (different UUIDs) — we pick the most recent one.

In [ ]:
def find_zip_for_ticker(ticker: str) -> Path | None:
    """Find the most recent zip file for a ticker (by modification time)."""
    candidates = [f for f in ASSET_INDICES_DIR.iterdir() if f.suffix == ".zip" and f.stem.endswith(f"_{ticker}")]
    if not candidates:
        return None
    return max(candidates, key=lambda f: f.stat().st_mtime)


def load_sentiment_indices(zip_path: Path) -> pd.DataFrame:
    """Load all hourly sentiment data from a zip file."""
    frames = []
    with zipfile.ZipFile(zip_path, "r") as zf:
        for name in zf.namelist():
            parts = name.split("/")
            if len(parts) < 3:
                continue
            idx_type = parts[0].replace("index_type=", "")
            topic = parts[1].replace("topic=", "")
            with zf.open(name) as zipped_file:
                with gzip.open(zipped_file) as gz:
                    df = pd.read_csv(gz)
            df["index_type"] = idx_type
            df["topic"] = topic
            frames.append(df)
    combined = pd.concat(frames, ignore_index=True)
    combined["publication_time"] = pd.to_datetime(combined["publication_time"])
    return combined


all_raw_indices = {}
for ticker in TICKERS:
    zp = find_zip_for_ticker(ticker)
    if zp is None:
        print(f"WARNING: No zip found for {ticker}")
        continue
    df = load_sentiment_indices(zp)
    all_raw_indices[ticker] = df
    print(
        f"{ticker}: {len(df):>9,} hourly rows | topics: {df['topic'].nunique()} | "
        f"index_types: {sorted(df['index_type'].unique())} | "
        f"date range: {df['publication_time'].min().date()} to {df['publication_time'].max().date()}"
    )

sample_ticker = TICKERS[0]
print(f"\nSample hourly data ({sample_ticker}):")
all_raw_indices[sample_ticker].head(8)

## Step 2: Load & Prepare Price Data

We load the merged price file, match each ticker, compute **forward returns** at multiple horizons (1d, 7d, 1m, 3m), and classify each day into an **UP or DOWN regime** based on the trailing 63-day return.

In [ ]:
def load_and_prepare_prices(ticker: str, prices: pd.DataFrame) -> pd.DataFrame | None:
    candidates = [
        ticker,
        ticker.replace("_", " "),
        ticker.split("_")[0] + " " + "_".join(ticker.split("_")[1:]),
    ]
    prices_ticker = None
    for c in candidates:
        subset = prices[prices["ticker"] == c]
        if not subset.empty:
            prices_ticker = subset.copy()
            break
    if prices_ticker is None:
        partial = [t for t in prices["ticker"].unique() if ticker.split("_")[0] in t]
        if partial:
            prices_ticker = prices[prices["ticker"] == partial[0]].copy()
    if prices_ticker is None:
        return None

    prices_ticker["trade_date"] = pd.to_datetime(prices_ticker["trade_date"])
    prices_ticker = prices_ticker.sort_values("trade_date").set_index("trade_date")
    price_col = "ask_close" if "ask_close" in prices_ticker.columns else "price"

    for label, days in RETURN_HORIZONS.items():
        prices_ticker[f"fwd_ret_{label}"] = prices_ticker[price_col].shift(-days) / prices_ticker[price_col] - 1

    trailing = prices_ticker[price_col] / prices_ticker[price_col].shift(REGIME_LOOKBACK) - 1
    prices_ticker["regime"] = np.where(trailing >= 0, "UP", "DOWN")
    prices_ticker.loc[trailing.isna(), "regime"] = np.nan
    return prices_ticker


raw_prices = pd.read_csv(PRICES_FILE)
all_prices = {}
for ticker in TICKERS:
    pt = load_and_prepare_prices(ticker, raw_prices)
    if pt is None:
        print(f"WARNING: No price data for {ticker}")
        continue
    all_prices[ticker] = pt
    rc = pt["regime"].value_counts()
    print(f"{ticker}: {len(pt)} price days | UP={rc.get('UP', 0)} DOWN={rc.get('DOWN', 0)}")

print(f"\nSample prices ({sample_ticker}):")
_pc = "ask_close" if "ask_close" in all_prices[sample_ticker].columns else "price"
all_prices[sample_ticker][[_pc, "fwd_ret_7d", "fwd_ret_1m", "regime"]].head(5)

## Step 3: Aggregate Hourly Indices to Daily Sentiment (6 Metrics)

The raw index data is at hourly granularity. We aggregate to daily, preserving all raw columns and deriving **6 daily metrics**:

| Metric | Formula | Measures |
|---|---|---|
| `sentiment_avg` | `sentiment_sum / headline_count` | Average direction |
| `sentiment_abs_avg` | `sentiment_abs_sum / headline_count` | Intensity (direction-agnostic) |
| `sentiment_std_daily` | mean of hourly `sentiment_std` | Intra-day disagreement |
| `headline_count` | sum of hourly counts | Attention / volume |
| `conviction_ratio` | `sentiment_sum / sentiment_abs_sum` | Net directional conviction |
| `intensity_per_headline` | `sentiment_abs_sum / headline_count` | Average strength per headline |

We then split into two topic levels:

- **GROUP**: the prefix before the first `-` (e.g. `Macroeconomic Indicators`, `Market Dynamics`)
- **TOPIC**: the full topic name (e.g. `Market Dynamics-Price Action & Volatility`)

### Coverage & sparse indices

Not every topic has data every day. We compute **coverage** — the percentage of business days with at least one observation. Indices with coverage below **15%** are flagged as **sparse** but still included in the analysis.

In [ ]:
def aggregate_to_daily(raw: pd.DataFrame) -> pd.DataFrame:
    """Aggregate hourly sentiment indices to daily, deriving all 6 metrics."""
    raw = raw.copy()
    raw["date"] = raw["publication_time"].dt.normalize()

    daily = (
        raw.groupby(["date", "index_type", "topic"])
        .agg(
            headline_count=("headline_count", "sum"),
            sentiment_sum=("sentiment_sum", "sum"),
            sentiment_abs_sum=("sentiment_abs_sum", "sum"),
            sentiment_std_mean=("sentiment_std", "mean"),
        )
        .reset_index()
    )
    daily["sentiment_avg"] = daily["sentiment_sum"] / daily["headline_count"]
    daily["sentiment_abs_avg"] = daily["sentiment_abs_sum"] / daily["headline_count"]
    daily["sentiment_std_daily"] = daily["sentiment_std_mean"]
    daily["conviction_ratio"] = daily["sentiment_sum"] / daily["sentiment_abs_sum"].replace(0, np.nan)
    daily["intensity_per_headline"] = daily["sentiment_abs_sum"] / daily["headline_count"]

    daily["group"] = daily["topic"].str.split("-").str[0]

    metric_cols = [
        "headline_count",
        "sentiment_sum",
        "sentiment_abs_sum",
        "sentiment_avg",
        "sentiment_abs_avg",
        "sentiment_std_daily",
        "conviction_ratio",
        "intensity_per_headline",
    ]

    group_agg = (
        daily.groupby(["date", "index_type", "group"])
        .agg(
            headline_count=("headline_count", "sum"),
            sentiment_sum=("sentiment_sum", "sum"),
            sentiment_abs_sum=("sentiment_abs_sum", "sum"),
            sentiment_std_daily=("sentiment_std_daily", "mean"),
        )
        .reset_index()
    )
    group_agg["sentiment_avg"] = group_agg["sentiment_sum"] / group_agg["headline_count"]
    group_agg["sentiment_abs_avg"] = group_agg["sentiment_abs_sum"] / group_agg["headline_count"]
    group_agg["conviction_ratio"] = group_agg["sentiment_sum"] / group_agg["sentiment_abs_sum"].replace(0, np.nan)
    group_agg["intensity_per_headline"] = group_agg["sentiment_abs_sum"] / group_agg["headline_count"]
    group_agg.rename(columns={"group": "name"}, inplace=True)
    group_agg["topic_level"] = "GROUP"

    keep = ["date", "index_type", "topic"] + [c for c in metric_cols if c in daily.columns]
    topic_part = daily[keep].copy()
    topic_part.rename(columns={"topic": "name"}, inplace=True)
    topic_part["topic_level"] = "TOPIC"

    common_cols = ["date", "index_type", "name", "topic_level"] + SENTIMENT_METRICS
    return pd.concat(
        [
            group_agg[[c for c in common_cols if c in group_agg.columns]],
            topic_part[[c for c in common_cols if c in topic_part.columns]],
        ],
        ignore_index=True,
    )


all_sentiment = {}
for ticker in TICKERS:
    if ticker not in all_raw_indices:
        continue
    sent = aggregate_to_daily(all_raw_indices[ticker])
    all_sentiment[ticker] = sent
    n_idx = sent.groupby(["topic_level", "index_type", "name"]).ngroups
    print(f"{ticker}: {n_idx} unique sentiment indices, {len(sent):,} daily rows")

print(f"\nSample daily sentiment ({sample_ticker}):")
display_cols = ["date", "index_type", "name", "topic_level"] + SENTIMENT_METRICS
all_sentiment[sample_ticker][display_cols].head(8)

## Step 4: Compute Coverage & Run Correlation Analysis

For every combination of `(regime, metric, topic_level, index_type, name, transform, window, return_horizon)` we:

1. Apply a **rolling transform** to the daily sentiment series
2. Align with forward price returns
3. Compute **Pearson correlation** and a two-tailed **t-test** p-value

### Why Pearson correlation?

- **r > 0**: Rising sentiment predicts rising prices (momentum signal)
- **r < 0**: Rising sentiment predicts falling prices (contrarian signal)
- **|r|**: Strength of the linear relationship
- **p-value**: Probability of observing this correlation under the null hypothesis (no relationship)

### Why these two transforms?

- **Rolling z-score** — normalises sentiment relative to its own recent history. Stationary and comparable across indices. This is the primary transform.
- **Rolling mean** — smooths out daily noise. Kept as a secondary check but may produce spurious correlations if sentiment series have trends.

In [ ]:
def compute_coverage(sentiment_all: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (tl, it, nm), grp in sentiment_all.groupby(["topic_level", "index_type", "name"]):
        bdays = len(pd.bdate_range(grp["date"].min(), grp["date"].max()))
        n_days = grp["date"].nunique()
        rows.append(
            {
                "topic_level": tl,
                "index_type": it,
                "name": nm,
                "days_with_data": n_days,
                "total_bdays": bdays,
                "coverage_pct": round(100 * n_days / max(bdays, 1), 1),
            }
        )
    return pd.DataFrame(rows)


def apply_transforms(series: pd.Series, window: int) -> dict[str, pd.Series]:
    roll_mean = series.rolling(window, min_periods=max(1, window // 2)).mean()
    roll_std = series.rolling(window, min_periods=max(1, window // 2)).std()
    return {
        "rolling_zscore": (series - roll_mean) / roll_std.replace(0, np.nan),
        "rolling_mean": roll_mean,
    }


def vectorized_correlation_sweep(sentiment_all: pd.DataFrame, prices_ticker: pd.DataFrame) -> pd.DataFrame:
    fwd_ret_cols = [f"fwd_ret_{h}" for h in RETURN_HORIZONS]
    horizon_names = list(RETURN_HORIZONS.keys())
    prices_indexed = prices_ticker[["regime"] + fwd_ret_cols].copy()
    regime_col = prices_indexed["regime"]
    ret_matrix = prices_indexed[fwd_ret_cols].values
    price_index = prices_indexed.index

    results = []
    for (topic_level, index_type, name), grp in sentiment_all.groupby(["topic_level", "index_type", "name"]):
        grp = grp.set_index("date").sort_index()
        full_idx = pd.date_range(grp.index.min(), grp.index.max(), freq="B")

        for metric in SENTIMENT_METRICS:
            if metric not in grp.columns:
                continue
            fill_val = 0 if metric != "conviction_ratio" else np.nan
            raw_series = grp[metric].reindex(full_idx).fillna(fill_val)
            if raw_series.dropna().nunique() < 3:
                continue

            for window in ROLLING_WINDOWS:
                transformed = apply_transforms(raw_series, window)
                for transform_name, t_series in transformed.items():
                    common_idx = t_series.dropna().index.intersection(price_index)
                    if len(common_idx) < 30:
                        continue
                    s_vals = t_series.loc[common_idx].values
                    p_loc = price_index.get_indexer(common_idx)
                    r_vals = ret_matrix[p_loc]
                    reg_vals = regime_col.values[p_loc]

                    for regime in REGIMES:
                        rmask = np.ones(len(common_idx), dtype=bool) if regime == "BOTH" else (reg_vals == regime)
                        s_reg, r_reg = s_vals[rmask], r_vals[rmask]
                        if rmask.sum() < 30:
                            continue
                        s_finite = np.isfinite(s_reg)
                        for h_idx, horizon in enumerate(horizon_names):
                            r_col = r_reg[:, h_idx]
                            valid = s_finite & np.isfinite(r_col)
                            n_h = valid.sum()
                            if n_h < 30:
                                continue
                            sx, ry = s_reg[valid], r_col[valid]
                            sx_m, ry_m = sx - sx.mean(), ry - ry.mean()
                            ss_x, ss_y = (sx_m**2).sum(), (ry_m**2).sum()
                            if ss_x == 0 or ss_y == 0:
                                continue
                            corr = np.clip((sx_m * ry_m).sum() / np.sqrt(ss_x * ss_y), -1, 1)
                            if abs(corr) < 1:
                                t_stat = corr * np.sqrt((n_h - 2) / (1 - corr**2))
                                pval = 2 * t_dist.sf(abs(t_stat), n_h - 2)
                            else:
                                t_stat, pval = np.inf, 0.0
                            results.append(
                                (
                                    regime,
                                    metric,
                                    topic_level,
                                    index_type,
                                    name,
                                    transform_name,
                                    window,
                                    horizon,
                                    int(n_h),
                                    round(corr, 6),
                                    pval,
                                    round(t_stat, 4),
                                )
                            )

    if not results:
        return pd.DataFrame()
    return pd.DataFrame(
        results,
        columns=[
            "regime",
            "metric",
            "topic_level",
            "index_type",
            "name",
            "transform",
            "window",
            "return_horizon",
            "n_obs",
            "pearson_r",
            "p_value",
            "t_stat",
        ],
    )


all_corr_results = {}
all_coverage = {}
for ticker in TICKERS:
    if ticker not in all_sentiment or ticker not in all_prices:
        continue
    print(f"\n--- {ticker} ---")
    t0 = time.perf_counter()
    cov = compute_coverage(all_sentiment[ticker])
    all_coverage[ticker] = cov
    res = vectorized_correlation_sweep(all_sentiment[ticker], all_prices[ticker])
    res["ticker"] = ticker
    cov_map = cov.set_index(["topic_level", "index_type", "name"])["coverage_pct"]
    res["coverage_pct"] = res.set_index(["topic_level", "index_type", "name"]).index.map(cov_map).values
    res["coverage_pct"] = res["coverage_pct"].fillna(0)
    all_corr_results[ticker] = res
    elapsed = time.perf_counter() - t0
    n_sparse = (cov["coverage_pct"] < SPARSE_THRESHOLD_PCT).sum()
    n_metrics_used = res["metric"].nunique() if not res.empty else 0
    print(
        f"  {len(res):,} correlation tests | {len(cov)} indices x {n_metrics_used} metrics ({n_sparse} sparse) | {elapsed:.1f}s"
    )

total = sum(len(v) for v in all_corr_results.values())
print(f"\nTotal tests across all tickers: {total:,} ({len(SENTIMENT_METRICS)} metrics per index)")

## Step 5: Apply Significance Testing (Benjamini-Hochberg FDR)

### The multiple-testing problem

With thousands of correlation tests per ticker, we expect many false positives at p < 0.05. For example, 8,000 tests × 5% = 400 expected false positives under the null.

### How we address it

1. **Raw p-value** threshold at 0.05 (labelled `significant`)
2. **Benjamini-Hochberg (BH) FDR correction** — applied per regime to control the false discovery rate (labelled `significant (BH adjusted)`)

### How to read the output

- **BH sig**: number of tests surviving the BH correction
- **Expected FP**: 5% × total tests (what we'd expect by chance)
- **Excess**: BH sig − Expected FP (positive = more signal than noise)

In [ ]:
def benjamini_hochberg(p_values: np.ndarray, alpha: float = 0.05) -> np.ndarray:
    n = len(p_values)
    if n == 0:
        return np.array([], dtype=bool)
    sorted_idx = np.argsort(p_values)
    sorted_p = p_values[sorted_idx]
    thresholds = alpha * np.arange(1, n + 1) / n
    below = sorted_p <= thresholds
    if not below.any():
        return np.zeros(n, dtype=bool)
    max_idx = np.max(np.where(below))
    significant = np.zeros(n, dtype=bool)
    significant[sorted_idx[: max_idx + 1]] = True
    return significant


def apply_significance(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    significance = pd.Series("not significant", index=df.index)
    significance[df["p_value"] < SIGNIFICANCE_LEVEL] = "significant"
    for regime in REGIMES:
        regime_idx = df.index[df["regime"] == regime]
        if len(regime_idx) == 0:
            continue
        bh_mask = benjamini_hochberg(df.loc[regime_idx, "p_value"].values, SIGNIFICANCE_LEVEL)
        significance.loc[regime_idx[bh_mask]] = "significant (BH adjusted)"
    df["significance"] = significance
    return df


for ticker in TICKERS:
    if ticker not in all_corr_results:
        continue
    all_corr_results[ticker] = apply_significance(all_corr_results[ticker])
    res = all_corr_results[ticker]
    total = len(res)
    n_bh = (res["significance"] == "significant (BH adjusted)").sum()
    expected_fp = total * SIGNIFICANCE_LEVEL
    print(
        f"\n{ticker}: {total:,} tests | BH sig: {n_bh:,} ({100 * n_bh / total:.1f}%) | "
        f"Expected FP: {expected_fp:.0f} | Excess: {n_bh - expected_fp:+.0f}"
    )
    bh = res[res["significance"] == "significant (BH adjusted)"]
    for metric in SENTIMENT_METRICS:
        m = bh[bh["metric"] == metric]
        m_total = (res["metric"] == metric).sum()
        max_r = m["pearson_r"].abs().max() if not m.empty else 0
        print(
            f"  {metric:<28s}: {len(m):>5,} BH-sig / {m_total:>5,} ({100 * len(m) / max(m_total, 1):.1f}%)  max|r|={max_r:.4f}"
        )

## Step 6: Build Index Summary & Walk-Forward Validation

For each `(topic_level, index_type, name)` we build a summary row with:

- Coverage stats and a sparse flag
- Per-regime best correlation, p-value, significant horizons, and **which metric** produced it
- Per-metric best |r| and significance count
- A **composite score** = Σ -log10(p) × |r| across regimes
- The **best_metric** — which of the 6 metrics produces the strongest single driver for this index

Then we run a **60/40 walk-forward split**: find significant drivers in the in-sample period (across all metrics), and check how many retain BH-significance out-of-sample. The walk-forward key includes the `metric` dimension so each metric is validated independently.

In [ ]:
def build_index_summary(results_df: pd.DataFrame, coverage_df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    sig = results_df[results_df["significance"] == "significant (BH adjusted)"].copy()
    if not sig.empty:
        sig["abs_r"] = sig["pearson_r"].abs()

    sig_grouped = {}
    if not sig.empty:
        for (tl, it, nm, regime), grp in sig.groupby(["topic_level", "index_type", "name", "regime"]):
            sig_grouped[(tl, it, nm, regime)] = grp

    horizon_order = list(RETURN_HORIZONS.keys())
    rows = []
    for _, cov_row in coverage_df.iterrows():
        tl, it, nm = cov_row["topic_level"], cov_row["index_type"], cov_row["name"]
        row = {
            "ticker": ticker,
            "topic_level": tl,
            "name": nm,
            "index_type": it,
            "days_with_data": cov_row["days_with_data"],
            "total_bdays": cov_row["total_bdays"],
            "coverage_pct": cov_row["coverage_pct"],
            "sparse": cov_row["coverage_pct"] < SPARSE_THRESHOLD_PCT,
        }
        score = 0.0
        for regime in REGIMES:
            r_sig = sig_grouped.get((tl, it, nm, regime))
            if r_sig is None or r_sig.empty:
                row[f"{regime}_sig_horizons"] = ""
                row[f"{regime}_best_r"] = np.nan
                row[f"{regime}_best_t"] = np.nan
                row[f"{regime}_best_p"] = np.nan
                row[f"{regime}_best_metric"] = ""
            else:
                horizons = sorted(
                    r_sig["return_horizon"].unique(),
                    key=lambda h: horizon_order.index(h),
                )
                row[f"{regime}_sig_horizons"] = "|".join(horizons)
                best_idx = r_sig["abs_r"].idxmax()
                best_r = r_sig.loc[best_idx, "pearson_r"]
                best_p = r_sig.loc[best_idx, "p_value"]
                row[f"{regime}_best_r"] = round(best_r, 4)
                row[f"{regime}_best_t"] = r_sig.loc[best_idx, "t_stat"]
                row[f"{regime}_best_p"] = best_p
                row[f"{regime}_best_metric"] = r_sig.loc[best_idx, "metric"]
                score += -np.log10(max(best_p, 1e-300)) * abs(best_r)

        sig_regimes = [r for r in REGIMES if row.get(f"{r}_sig_horizons", "") != ""]
        row["sig_regimes"] = "|".join(sig_regimes) if sig_regimes else ""

        idx_sig = (
            sig[(sig["topic_level"] == tl) & (sig["index_type"] == it) & (sig["name"] == nm)]
            if not sig.empty
            else pd.DataFrame()
        )
        best_metric = ""
        best_abs_r = 0
        for metric in SENTIMENT_METRICS:
            m_sig = idx_sig[idx_sig["metric"] == metric] if not idx_sig.empty else pd.DataFrame()
            row[f"{metric}_n_sig"] = len(m_sig)
            if not m_sig.empty:
                m_best_idx = m_sig["abs_r"].idxmax()
                row[f"{metric}_best_r"] = round(m_sig.loc[m_best_idx, "pearson_r"], 4)
                if m_sig.loc[m_best_idx, "abs_r"] > best_abs_r:
                    best_abs_r = m_sig.loc[m_best_idx, "abs_r"]
                    best_metric = metric
            else:
                row[f"{metric}_best_r"] = np.nan
        row["best_metric"] = best_metric

        row["score"] = round(score, 2)
        rows.append(row)

    summary = pd.DataFrame(rows)
    summary.sort_values("score", ascending=False, inplace=True)
    summary.insert(0, "rank", range(1, len(summary) + 1))
    return summary


def walk_forward_validate(sentiment_all: pd.DataFrame, prices_ticker: pd.DataFrame, ticker: str) -> pd.DataFrame:
    all_dates = prices_ticker.index.sort_values()
    n_total = len(all_dates)
    n_is = int(n_total * WALK_FORWARD_IS_FRACTION)
    split_date = all_dates[n_is - 1]
    is_start, is_end = all_dates[0], split_date
    oos_start, oos_end = all_dates[n_is], all_dates[-1]

    is_sent = sentiment_all[(sentiment_all["date"] >= is_start) & (sentiment_all["date"] <= is_end)]
    is_prices = prices_ticker.loc[is_start:is_end]
    is_results = apply_significance(vectorized_correlation_sweep(is_sent, is_prices))
    if is_results.empty:
        return pd.DataFrame()

    oos_sent = sentiment_all[(sentiment_all["date"] >= oos_start) & (sentiment_all["date"] <= oos_end)]
    oos_prices = prices_ticker.loc[oos_start:oos_end]
    oos_results = apply_significance(vectorized_correlation_sweep(oos_sent, oos_prices))
    if oos_results.empty:
        return pd.DataFrame()

    def _key(row):
        return (
            row["regime"],
            row["metric"],
            row["topic_level"],
            row["index_type"],
            row["name"],
            row["transform"],
            row["window"],
            row["return_horizon"],
        )

    is_sig = is_results[is_results["significance"] == "significant (BH adjusted)"].copy()
    is_sig["key"] = is_sig.apply(_key, axis=1)
    oos_results["key"] = oos_results.apply(_key, axis=1)
    oos_lookup = {row["key"]: row for _, row in oos_results.iterrows()}

    matched = []
    for _, is_row in is_sig.iterrows():
        k = is_row["key"]
        if k in oos_lookup:
            oos_row = oos_lookup[k]
            matched.append(
                {
                    "regime": is_row["regime"],
                    "metric": is_row["metric"],
                    "topic_level": is_row["topic_level"],
                    "index_type": is_row["index_type"],
                    "name": is_row["name"],
                    "transform": is_row["transform"],
                    "window": is_row["window"],
                    "return_horizon": is_row["return_horizon"],
                    "is_r": is_row["pearson_r"],
                    "is_p": is_row["p_value"],
                    "oos_r": oos_row["pearson_r"],
                    "oos_p": oos_row["p_value"],
                    "oos_sig": oos_row["significance"],
                }
            )
    return pd.DataFrame(matched)


def enrich_with_walkforward(idx_summary: pd.DataFrame, wf_df: pd.DataFrame) -> pd.DataFrame:
    idx_summary = idx_summary.copy()
    for col, val in {
        "wf_is_sig": 0,
        "wf_oos_persistent": 0,
        "wf_oos_pct": 0.0,
        "wf_same_sign": 0,
        "wf_sign_rate": 0.0,
        "wf_best_oos_r": np.nan,
        "wf_best_oos_p": np.nan,
    }.items():
        idx_summary[col] = val
    if wf_df.empty:
        return idx_summary
    for (tl, it, nm), grp in wf_df.groupby(["topic_level", "index_type", "name"]):
        mask = (idx_summary["topic_level"] == tl) & (idx_summary["index_type"] == it) & (idx_summary["name"] == nm)
        if not mask.any():
            continue
        n_is = len(grp)
        persistent = grp[grp["oos_sig"] == "significant (BH adjusted)"]
        n_oos = len(persistent)
        same_sign = (np.sign(grp["is_r"]) == np.sign(grp["oos_r"])).sum()
        idx_summary.loc[mask, "wf_is_sig"] = n_is
        idx_summary.loc[mask, "wf_oos_persistent"] = n_oos
        idx_summary.loc[mask, "wf_oos_pct"] = round(100 * n_oos / max(n_is, 1), 1)
        idx_summary.loc[mask, "wf_same_sign"] = same_sign
        idx_summary.loc[mask, "wf_sign_rate"] = round(100 * same_sign / max(n_is, 1), 1)
        if not persistent.empty:
            best_idx = persistent["oos_r"].abs().idxmax()
            idx_summary.loc[mask, "wf_best_oos_r"] = round(persistent.loc[best_idx, "oos_r"], 4)
            idx_summary.loc[mask, "wf_best_oos_p"] = persistent.loc[best_idx, "oos_p"]
    return idx_summary


all_idx_summaries = {}
for ticker in TICKERS:
    if ticker not in all_corr_results:
        continue
    print(f"\n--- {ticker} ---")
    t0 = time.perf_counter()
    idx_sum = build_index_summary(all_corr_results[ticker], all_coverage[ticker], ticker)
    n_sig = (idx_sum["score"] > 0).sum()
    print(f"  Index summary: {len(idx_sum)} indices, {n_sig} significant")

    print(f"  Running walk-forward validation...")
    wf_df = walk_forward_validate(all_sentiment[ticker], all_prices[ticker], ticker)
    n_is = len(wf_df)
    n_oos = (wf_df["oos_sig"] == "significant (BH adjusted)").sum() if not wf_df.empty else 0
    elapsed = time.perf_counter() - t0
    print(f"  WF: {n_is} IS-sig matched, {n_oos} persistent OOS ({100 * n_oos / max(n_is, 1):.1f}%) [{elapsed:.1f}s]")

    idx_sum = enrich_with_walkforward(idx_sum, wf_df)
    all_idx_summaries[ticker] = idx_sum

print(f"\nTop 5 drivers for {sample_ticker}:")
all_idx_summaries[sample_ticker][
    [
        "rank",
        "name",
        "index_type",
        "score",
        "best_metric",
        "BOTH_best_r",
        "wf_oos_persistent",
    ]
].head(5)

## Step 7: Cross-Ticker Assessment

We now aggregate results across all tickers into three summary tables:

1. **Overall Ranking** — composite grade (A-F) based on BH significance, correlation strength, OOS persistence, and **dominant metric** per ticker
2. **Top Sentiment Drivers Per Ticker** — the single highest-scoring topic for each asset, showing which metric produced it
3. **Top Sentiment Group Drivers Per Ticker** — group-level aggregation showing breadth of signal and best metric per group

In [ ]:
def load_correlation_summary_from_idx(ticker: str, idx: pd.DataFrame, results_df: pd.DataFrame) -> dict:
    n_indices = len(idx)
    n_sig = (idx["score"] > 0).sum()
    n_sparse_sig = ((idx["sparse"]) & (idx["score"] > 0)).sum()

    best_r_both = idx["BOTH_best_r"].abs().max() if "BOTH_best_r" in idx else np.nan
    best_r_up = idx["UP_best_r"].abs().max() if "UP_best_r" in idx else np.nan
    best_r_down = idx["DOWN_best_r"].abs().max() if "DOWN_best_r" in idx else np.nan
    best_r_any = np.nanmax([best_r_both, best_r_up, best_r_down])

    top_score = idx["score"].max()
    top_row = idx.loc[idx["score"].idxmax()] if top_score > 0 else None
    top_driver = top_row["name"] if top_row is not None else "--"
    top_idx_type = top_row["index_type"] if top_row is not None else "--"
    top_best_metric = top_row["best_metric"] if top_row is not None and "best_metric" in idx else "--"

    sig_idx = idx[idx["score"] > 0]
    metric_wins = sig_idx["best_metric"].value_counts() if "best_metric" in sig_idx else pd.Series(dtype=int)
    dominant_metric = metric_wins.index[0] if len(metric_wins) > 0 else "--"

    total_tests = len(results_df)
    n_bh = (results_df["significance"] == "significant (BH adjusted)").sum()

    wf_is = int(idx["wf_is_sig"].sum()) if "wf_is_sig" in idx else 0
    wf_oos = int(idx["wf_oos_persistent"].sum()) if "wf_oos_persistent" in idx else 0
    wf_pct = round(100 * wf_oos / max(wf_is, 1), 1) if wf_is > 0 else 0.0
    wf_sign = int(idx["wf_same_sign"].sum()) if "wf_same_sign" in idx else 0
    wf_sign_pct = round(100 * wf_sign / max(wf_is, 1), 1) if wf_is > 0 else 0.0

    best_oos_r = np.nan
    if "wf_best_oos_r" in idx:
        valid = idx["wf_best_oos_r"].dropna()
        if not valid.empty:
            best_oos_r = valid.abs().max()

    sig_regimes = idx[idx["score"] > 0]["sig_regimes"].value_counts()
    n_all_regime = sum(1 for v in sig_regimes.index if "BOTH" in v and "UP" in v and "DOWN" in v)
    n_regime_specific = n_sig - n_all_regime

    return {
        "ticker": ticker,
        "n_indices": n_indices,
        "total_corr_tests": total_tests,
        "n_bh_sig": n_bh,
        "bh_sig_pct": round(100 * n_bh / max(total_tests, 1), 1),
        "n_sig_indices": n_sig,
        "n_sparse_sig": n_sparse_sig,
        "top_score": top_score,
        "top_driver": top_driver,
        "top_idx_type": top_idx_type,
        "top_best_metric": top_best_metric,
        "dominant_metric": dominant_metric,
        "best_r_any": round(best_r_any, 4) if not np.isnan(best_r_any) else np.nan,
        "best_r_both": round(best_r_both, 4) if not np.isnan(best_r_both) else np.nan,
        "wf_is_sig": wf_is,
        "wf_oos_persistent": wf_oos,
        "wf_oos_pct": wf_pct,
        "wf_sign_consistency": wf_sign_pct,
        "best_oos_r": round(best_oos_r, 4) if not np.isnan(best_oos_r) else np.nan,
        "n_all_regime_sig": n_all_regime,
        "n_regime_specific": n_regime_specific,
    }


def compute_overall_grade(row: dict) -> tuple[str, float]:
    score = 0.0
    score += min(row.get("bh_sig_pct", 0) / 30.0, 1.0) * 10
    score += min(row.get("n_sig_indices", 0) / 80.0, 1.0) * 10
    score += min(row.get("top_score", 0) / 50.0, 1.0) * 15
    best_r = row.get("best_r_any", 0)
    if np.isnan(best_r):
        best_r = 0
    score += min(best_r / 0.5, 1.0) * 10
    score += min(row.get("wf_oos_pct", 0) / 50.0, 1.0) * 30
    score += min(row.get("wf_sign_consistency", 0) / 80.0, 1.0) * 10
    oos_r = row.get("best_oos_r", 0)
    if np.isnan(oos_r):
        oos_r = 0
    score += min(oos_r / 0.5, 1.0) * 15
    grade = "A" if score >= 80 else "B" if score >= 65 else "C" if score >= 45 else "D" if score >= 25 else "F"
    return grade, round(score, 1)


def load_group_drivers(ticker: str, idx: pd.DataFrame) -> pd.DataFrame:
    sig = idx[idx["score"] > 0].copy()
    if sig.empty:
        return pd.DataFrame()
    sig["group"] = sig["name"].str.split("-").str[0].str.strip()
    rows = []
    for group, grp in sig.groupby("group"):
        best_r_cols = [c for c in ["BOTH_best_r", "UP_best_r", "DOWN_best_r"] if c in grp]
        best_r = grp[best_r_cols].abs().max().max() if best_r_cols else np.nan
        best_score_idx = grp["score"].idxmax()
        best_name = grp.loc[best_score_idx, "name"]
        best_idx_type = grp.loc[best_score_idx, "index_type"]
        best_metric = grp.loc[best_score_idx, "best_metric"] if "best_metric" in grp else "--"
        wf_oos = int(grp["wf_oos_persistent"].sum()) if "wf_oos_persistent" in grp else 0
        wf_is = int(grp["wf_is_sig"].sum()) if "wf_is_sig" in grp else 0
        wf_pct = round(100 * wf_oos / max(wf_is, 1), 1) if wf_is > 0 else 0.0
        best_oos_r = np.nan
        if "wf_best_oos_r" in grp:
            valid = grp["wf_best_oos_r"].dropna()
            if not valid.empty:
                best_oos_r = valid.abs().max()
        rows.append(
            {
                "ticker": ticker,
                "label": TICKER_LABELS.get(ticker, ticker),
                "group": group,
                "n_sig": len(grp),
                "top_score": grp["score"].max(),
                "best_r": round(best_r, 4) if not np.isnan(best_r) else np.nan,
                "best_driver": best_name,
                "best_idx_type": best_idx_type,
                "best_metric": best_metric,
                "wf_is_sig": wf_is,
                "wf_oos_persistent": wf_oos,
                "wf_oos_pct": wf_pct,
                "best_oos_r": round(best_oos_r, 4) if not np.isnan(best_oos_r) else np.nan,
            }
        )
    return pd.DataFrame(rows).sort_values("top_score", ascending=False)


ct_rows = []
for ticker in TICKERS:
    if ticker not in all_idx_summaries:
        continue
    row = load_correlation_summary_from_idx(ticker, all_idx_summaries[ticker], all_corr_results[ticker])
    row["label"] = TICKER_LABELS.get(ticker, ticker)
    grade, score = compute_overall_grade(row)
    row["grade"] = grade
    row["overall_score"] = score
    ct_rows.append(row)

ct_df = pd.DataFrame(ct_rows)
ct_df.sort_values("overall_score", ascending=False, inplace=True)
ct_df.reset_index(drop=True, inplace=True)
ct_df.insert(0, "rank", range(1, len(ct_df) + 1))

group_frames = []
for ticker in TICKERS:
    if ticker not in all_idx_summaries:
        continue
    gdf = load_group_drivers(ticker, all_idx_summaries[ticker])
    if not gdf.empty:
        group_frames.append(gdf)
all_groups = pd.concat(group_frames, ignore_index=True) if group_frames else pd.DataFrame()

print("Cross-ticker assessment built successfully.")
print(f"Tickers: {len(ct_df)}, Group driver rows: {len(all_groups)}")

### Table 1: Overall Ranking

Each ticker receives a composite grade (A–F) based on:

- **BH %**: Percentage of tests that are BH-significant (signal density)
- **Sig Idx**: Number of sentiment indices with at least one significant result
- **Best |r|**: Strongest absolute correlation across all regimes
- **OOS %**: Percentage of IS-significant drivers that persist out-of-sample
- **Sign %**: Percentage of matched drivers with consistent sign IS vs OOS
- **OOS |r|**: Strongest OOS correlation among persistent drivers

In [ ]:
display_cols_1 = [
    "rank",
    "ticker",
    "label",
    "grade",
    "overall_score",
    "n_sig_indices",
    "bh_sig_pct",
    "best_r_any",
    "wf_oos_pct",
    "wf_sign_consistency",
    "best_oos_r",
    "dominant_metric",
]
ct_df[display_cols_1].rename(
    columns={
        "rank": "#",
        "label": "Asset",
        "grade": "Grade",
        "overall_score": "Score",
        "n_sig_indices": "Sig Idx",
        "bh_sig_pct": "BH %",
        "best_r_any": "Best |r|",
        "wf_oos_pct": "OOS %",
        "wf_sign_consistency": "Sign %",
        "best_oos_r": "OOS |r|",
    }
)

### Table 2: Top Sentiment Drivers Per Ticker

The single highest-scoring sentiment topic for each asset, with its composite score and correlation strength.

In [ ]:
display_cols_2 = [
    "rank",
    "ticker",
    "top_score",
    "top_driver",
    "top_idx_type",
    "top_best_metric",
    "best_r_both",
]
ct_df[display_cols_2].rename(
    columns={
        "rank": "#",
        "top_score": "Top Score",
        "top_driver": "Top Correlation Driver",
        "top_idx_type": "Index Type",
        "best_r_both": "|r| All",
    }
)

### Table 3: Top Sentiment Group Drivers Per Ticker

Group-level aggregation (prefix before the first `-`) showing the breadth and depth of signal within each sentiment category.

In [ ]:
if not all_groups.empty:
    display_cols_3 = [
        "ticker",
        "label",
        "group",
        "n_sig",
        "top_score",
        "best_r",
        "best_metric",
        "best_driver",
        "wf_oos_persistent",
        "wf_oos_pct",
        "best_oos_r",
    ]
    ipy_display(
        all_groups[display_cols_3].rename(
            columns={
                "label": "Asset",
                "n_sig": "Sig",
                "top_score": "Score",
                "best_r": "Best |r|",
                "best_driver": "Best Driver",
                "wf_oos_persistent": "OOS Sig",
                "wf_oos_pct": "OOS %",
                "best_oos_r": "OOS |r|",
            }
        )
    )
else:
    print("No group drivers found.")

### Regime Insights

Some sentiment drivers are significant across all market conditions, while others only work in **UP** (bullish) or **DOWN** (bearish) regimes.

- **Regime-universal drivers** are the most robust — they work regardless of the market environment
- **Regime-specific drivers** (UP-only or DOWN-only) still carry signal but should be deployed conditionally
- **Sign-flipping drivers** — where the correlation reverses between UP and DOWN regimes — are particularly interesting for regime-conditional strategies

In [ ]:
regime_rows = []
for ticker in TICKERS:
    if ticker not in all_idx_summaries:
        continue
    idx = all_idx_summaries[ticker]
    sig = idx[idx["score"] > 0].copy()
    if sig.empty:
        continue

    up_only = sig[sig["sig_regimes"].str.contains("UP", na=False) & ~sig["sig_regimes"].str.contains("DOWN", na=False)]
    down_only = sig[
        sig["sig_regimes"].str.contains("DOWN", na=False) & ~sig["sig_regimes"].str.contains("UP", na=False)
    ]
    both_ud = sig[sig["sig_regimes"].str.contains("UP", na=False) & sig["sig_regimes"].str.contains("DOWN", na=False)]

    n_sign_flip = 0
    flip_examples = []
    for _, row in both_ud.iterrows():
        up_r = row.get("UP_best_r")
        down_r = row.get("DOWN_best_r")
        if pd.notna(up_r) and pd.notna(down_r) and up_r * down_r < 0:
            n_sign_flip += 1
            if len(flip_examples) < 2:
                flip_examples.append(f"{row['name']} (UP r={up_r:+.3f}, DOWN r={down_r:+.3f})")

    regime_rows.append(
        {
            "ticker": ticker,
            "label": TICKER_LABELS.get(ticker, ticker),
            "total_sig": len(sig),
            "UP only": len(up_only),
            "DOWN only": len(down_only),
            "both UP & DOWN": len(both_ud),
            "sign flips": n_sign_flip,
            "flip_examples": "; ".join(flip_examples) if flip_examples else "--",
        }
    )

regime_df = pd.DataFrame(regime_rows)

print("Regime Breakdown: How many significant drivers are regime-specific vs universal?\n")
ipy_display(
    regime_df[
        [
            "ticker",
            "label",
            "total_sig",
            "UP only",
            "DOWN only",
            "both UP & DOWN",
            "sign flips",
        ]
    ].rename(columns={"label": "Asset", "total_sig": "Total Sig"})
)

flips = regime_df[regime_df["sign flips"] > 0]
if not flips.empty:
    print("\nSign-Flipping Drivers (correlation reverses between UP and DOWN regimes):\n")
    for _, row in flips.iterrows():
        print(f"  {row['ticker']} ({row['label']}): {row['sign flips']} driver(s)")
        if row["flip_examples"] != "--":
            for ex in row["flip_examples"].split("; "):
                print(f"    \u2192 {ex}")
else:
    print("\nNo sign-flipping drivers detected across any ticker.")